<a href="https://colab.research.google.com/github/alsulami412-arch/Vibration-group/blob/main/Vibration_group.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Home page - Vibration Group
from IPython.display import HTML, display
from datetime import date

profile = {
"Name": "Mohammed Alsulami",
"Employee ID": "87216",
"Position": "Vibration Analyst",
"Company": "Saudi Energy",
}

cards = "".join(
f"<div style='background:#ffffff14;border:1px solid #ffffff26;border-radius:10px;padding:12px 16px;min-width:160px'>"
f"<div style='font-size:11px;letter-spacing:1px;text-transform:uppercase;color:#9fd8d2'>{k}</div>"
f"<div style='font-size:17px;font-weight:600;margin-top:4px'>{v}</div></div>"
for k, v in profile.items()
)

html = f"""<div style='font-family:Segoe UI,Arial,sans-serif;background:linear-gradient(135deg,#0b1f33,#12476b);color:#fff;border-radius:14px;padding:28px 32px'>
<div style='font-size:12px;letter-spacing:2px;color:#9fd8d2'>SAUDI ENERGY - CONDITION MONITORING</div>
<h1 style='margin:6px 0 4px;font-size:30px;color:#fff'>Vibration Group</h1>
<div style='color:#c9d6e2;font-size:14px'>Vibration analysis workspace - {date.today():%d %B %Y}</div>
<div style='display:flex;flex-wrap:wrap;gap:12px;margin-top:20px'>{cards}</div>
</div>"""

display(HTML(html))

## Vibration Reports – Before T&I
Data source: `Vibration Reports Before T&I.xlsx` in this repository (Unit 03, Unit 04 and Common PH#2).
Condition colours are taken from the report (ISO 20816 assessment):
🟢 Green = Accepted · 🟡 Yellow = Alert · 🟠 Orange = Warning · 🔴 Red = Danger · ⚪ Not Evaluated

In [ ]:
# 1) Load the Excel report from GitHub and build one clean table
import os, re, urllib.request
import pandas as pd
import openpyxl

FILE = "Vibration Reports Before T&I.xlsx"
URL = ("https://raw.githubusercontent.com/alsulami412-arch/Vibration-group/main/"
       "Vibration%20Reports%20Before%20T%26I.xlsx")
if not os.path.exists(FILE):
    urllib.request.urlretrieve(URL, FILE)

AREA_NAMES = {
    "Vibration Report U#04": "Unit 04",
    "Vibration Report U#03": "Unit 03",
    "Vibration Report Common PH#2": "Common PH#2",
}
RGB_TO_CONDITION = {"FF00B050": "Green", "FFFFFF00": "Yellow", "FFFF0000": "Red"}
THEME_TO_CONDITION = {9: "Orange", 0: "Not Evaluated"}

def cell_condition(cell):
    fc = cell.fill.fgColor
    if fc.type == "rgb":
        return RGB_TO_CONDITION.get(fc.rgb, "Not Evaluated")
    if fc.type == "theme":
        return THEME_TO_CONDITION.get(fc.theme, "Not Evaluated")
    return "Not Evaluated"

READING = re.compile(r"(Motor|Pump|Fan)\s+(NDE|DE)\s+([HVA])\s*=\s*([\d.]+)\s*mm/s", re.I)

def parse_readings(text):
    """'Motor NDE H= 2.4 mm/s, Motor DE H= 1.6 mm/s' -> list of (point, value)"""
    out = []
    for comp, end, direction, val in READING.findall(str(text)):
        out.append((f"{comp.title()} {end.upper()} {direction.upper()}", float(val)))
    return out

wb = openpyxl.load_workbook(FILE)
rows = []
for ws in wb.worksheets:
    area = AREA_NAMES.get(ws.title, ws.title)
    # find the header row ("No.")
    header_row = next(r for r in range(1, ws.max_row + 1) if str(ws.cell(r, 1).value).strip() == "No.")
    for r in range(header_row + 1, ws.max_row + 1):
        no = ws.cell(r, 1).value
        if no is None or str(no).strip() == "":
            continue
        vib_text = ws.cell(r, 5).value
        readings = parse_readings(vib_text)
        rows.append({
            "Area": area,
            "No.": int(no),
            "Equipment": " ".join(str(ws.cell(r, 2).value).split()),
            "Functional Location": ws.cell(r, 3).value,
            "Finding": ws.cell(r, 4).value,
            "Vibration": vib_text,
            "Recommendation": ws.cell(r, 6).value,
            "Condition": cell_condition(ws.cell(r, 4)),
            "Max (mm/s)": max((v for _, v in readings), default=None),
            "Max point": max(readings, key=lambda x: x[1])[0] if readings else None,
        })

df = pd.DataFrame(rows)
CONDITION_ORDER = ["Green", "Yellow", "Orange", "Red", "Not Evaluated"]
df["Condition"] = pd.Categorical(df["Condition"], CONDITION_ORDER, ordered=True)
print(f"Loaded {len(df)} equipment items from {df['Area'].nunique()} areas")
df.head()

### Condition summary by area

In [ ]:
# 2) Summary dashboard
from IPython.display import HTML, display

COLORS = {"Green": "#00B050", "Yellow": "#FFD600", "Orange": "#F79646",
          "Red": "#E53935", "Not Evaluated": "#B0BEC5"}

summary = (df.pivot_table(index="Area", columns="Condition", values="No.",
                          aggfunc="count", fill_value=0, observed=False)
             .reindex(columns=CONDITION_ORDER, fill_value=0))
summary["Total"] = summary.sum(axis=1)
summary.loc["All areas"] = summary.sum()

total = len(df)
evaluated = int((df["Condition"] != "Not Evaluated").sum())
critical = int(df["Condition"].isin(["Orange", "Red"]).sum())
kpis = [("Equipment", total), ("Evaluated", evaluated),
        ("Accepted (Green)", int((df["Condition"] == "Green").sum())),
        ("Action needed (Orange/Red)", critical)]
cards = "".join(
    f"<div style='background:#ffffff14;border:1px solid #ffffff26;border-radius:10px;padding:12px 16px;min-width:150px'>"
    f"<div style='font-size:11px;letter-spacing:1px;text-transform:uppercase;color:#9fd8d2'>{k}</div>"
    f"<div style='font-size:24px;font-weight:700;margin-top:4px'>{v}</div></div>" for k, v in kpis)
display(HTML(f"<div style='font-family:Segoe UI,Arial,sans-serif;background:linear-gradient(135deg,#0b1f33,#12476b);"
             f"color:#fff;border-radius:14px;padding:20px 24px;display:flex;flex-wrap:wrap;gap:12px'>{cards}</div>"))

def header_color(col):
    return [f"background-color:{COLORS[col]}22" if col in COLORS else "" for _ in summary.index]

display(summary.style.apply(lambda s: header_color(s.name), axis=0)
               .set_caption("Number of equipment per condition"))

In [ ]:
# 3) Charts
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 1.4]})

# a) stacked bar – condition per area
s = summary.drop(index="All areas")[CONDITION_ORDER]
left = pd.Series(0, index=s.index)
for cond in CONDITION_ORDER:
    axes[0].barh(s.index, s[cond], left=left, color=COLORS[cond], label=cond, edgecolor="white")
    for i, (l, v) in enumerate(zip(left, s[cond])):
        if v:
            axes[0].text(l + v / 2, i, int(v), ha="center", va="center", fontsize=9)
    left += s[cond]
axes[0].set_title("Condition by area")
axes[0].set_xlabel("Number of equipment")
axes[0].legend(loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=5, frameon=False, fontsize=8)
axes[0].invert_yaxis()

# b) top 15 highest readings
top = df.dropna(subset=["Max (mm/s)"]).nlargest(15, "Max (mm/s)").iloc[::-1]
labels = top["Area"] + " | " + top["Equipment"]
axes[1].barh(labels, top["Max (mm/s)"], color=[COLORS[c] for c in top["Condition"]], edgecolor="#333", linewidth=0.3)
for y, (v, p) in enumerate(zip(top["Max (mm/s)"], top["Max point"])):
    axes[1].text(v, y, f" {v:g} ({p})", va="center", fontsize=8)
axes[1].set_xscale("log")
axes[1].set_title("Top 15 highest vibration readings (mm/s, log scale)")
axes[1].set_xlabel("Max overall velocity (mm/s)")
axes[1].tick_params(axis="y", labelsize=8)
axes[1].set_xlim(1, top["Max (mm/s)"].max() * 4)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Action list – Red and Orange equipment

In [ ]:
# 4) Equipment that needs action, highest vibration first
action = (df[df["Condition"].isin(["Red", "Orange"])]
            .sort_values(["Condition", "Max (mm/s)"], ascending=[False, False])
            [["Area", "Equipment", "Functional Location", "Condition", "Max (mm/s)",
              "Max point", "Finding", "Recommendation"]]
            .reset_index(drop=True))
action.index += 1

def color_row(row):
    return [f"background-color:{COLORS[row['Condition']]}33"] * len(row)

display(action.style.apply(color_row, axis=1).format({"Max (mm/s)": "{:g}"}))
print(f"{len(action)} items need action "
      f"({(action['Condition'] == 'Red').sum()} Red, {(action['Condition'] == 'Orange').sum()} Orange)")

### Search equipment

In [ ]:
# 5) Filter: change these values and re-run the cell
AREA = "All"          # "All", "Unit 03", "Unit 04", "Common PH#2"
CONDITION = "All"     # "All", "Green", "Yellow", "Orange", "Red", "Not Evaluated"
KEYWORD = ""          # e.g. "Condensate", "Fan", "W823"

view = df.copy()
if AREA != "All":
    view = view[view["Area"] == AREA]
if CONDITION != "All":
    view = view[view["Condition"] == CONDITION]
if KEYWORD:
    mask = view[["Equipment", "Functional Location", "Finding"]].apply(
        lambda c: c.astype(str).str.contains(KEYWORD, case=False)).any(axis=1)
    view = view[mask]

display(view.drop(columns=["Max point"]).style.apply(color_row, axis=1)
            .format({"Max (mm/s)": lambda v: "" if pd.isna(v) else f"{v:g}"}))
print(f"{len(view)} rows")

In [ ]:
# 6) Export the clean table and action list to Excel (downloads in Colab)
OUT = "Vibration_Analysis_Before_TI.xlsx"
with pd.ExcelWriter(OUT) as xw:
    summary.to_excel(xw, sheet_name="Summary")
    action.to_excel(xw, sheet_name="Action list")
    df.to_excel(xw, sheet_name="All equipment", index=False)
print("Saved", OUT)
try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass